# 02 — Baseline: TF-IDF + Logistic Regression

One pipeline per language. Writes metrics into `results/tables/all_results.csv` and confusion matrices into `results/confusion_matrices/`.

Also exports the top fake/real coefficient features per language for the explainability chapter.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src import config as C
from src.data_utils import get_split
from src.evaluate import evaluate_and_log
from src.models import baseline

In [ ]:
for lang in C.LANGUAGES:
    print(f'\n=== EN baseline :: {lang.upper()} ===')
    try:
        tr, va, te = get_split(lang)
    except FileNotFoundError as e:
        print(f'  skipped: {e}'); continue
    pipe = baseline.train(tr, va, lang)
    y_pred = baseline.predict(pipe, te, lang)
    metrics = evaluate_and_log(te['label'].values, y_pred, model_name='baseline', lang=lang)
    print('  metrics:', {k: round(v, 4) for k, v in metrics.items()})
    baseline.save(pipe, C.RESULTS_DIR / 'checkpoints' / f'baseline_{lang}.joblib')

### Top features (free explainability for the baseline)

In [ ]:
import pandas as pd
from src.models.baseline import top_features, load
for lang in C.LANGUAGES:
    p = C.RESULTS_DIR / 'checkpoints' / f'baseline_{lang}.joblib'
    if not p.exists(): continue
    pipe = load(p)
    fake_top, real_top = top_features(pipe, top_k=20)
    fake_top.to_csv(C.EXPLAIN_DIR / f'lr_top_fake_{lang}.csv', index=False)
    real_top.to_csv(C.EXPLAIN_DIR / f'lr_top_real_{lang}.csv', index=False)
    print(f'\n[{lang}] top fake-pulling tokens:'); display(fake_top.head(10))